# European Flight Delay Prediction — Exploratory Data Analysis

This notebook takes the sample of 100,000 previously created from the Flight Delays 2023–2025 Parquet raw dataset from Hugging Face and stored in CGS bucket and does some data analysis

The sample dataset was created in /notebooks/02_analysis_and_downsampling.ipynb

For reproducability we are capturing and storing the following
* CGS generation version of sample dataset used
* SHA-256 checksum finger prints for the sample dataset


## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [ ]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [ ]:
%pip install -r requirements.txt

## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME

If you want to reproduct a previous run - you can get these details from the provenance.json file in the appropriate processed/splits/runs/* folder
*   SOURCE_GENERATION - update from `None` to a previously recorded generation value of the processed/flights_downsampled_100k.parquet from the run provenance.json input.generation value
* EXPECTED_SOURCE_SHA256 - If you supply SOURCE_GENERATION and want to verify it has expected SHA-256 fingerprint update from `None` to the run provenance.json input.sha256 value






In [ ]:
from google.colab import auth
auth.authenticate_user()

import hashlib
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import sklearn
from google.cloud import storage
from sklearn.model_selection import train_test_split, StratifiedKFold

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"
SOURCE_OBJECT = "processed/flights_downsampled_100k.parquet"
SOURCE_GENERATION = None
EXPECTED_SOURCE_SHA256 = None
RANDOM_STATE = 42
TARGET_COLUMN = "delayed_15min"
SOURCE_YEAR_COLUMN = "source_year"
EXPECTED_DURATION_MIN_COLUMN = "expected_duration_min"

WORK_DIR = Path("/content/analysis")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE = WORK_DIR / "flights_downsampled_100k.parquet"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

## 3. Download processed dataset
Capture and download the specific version of the processed dataset from CGS based on the SOURCE_GENERATION

In [ ]:
if SOURCE_GENERATION is None:
    latest_blob = bucket.blob(SOURCE_OBJECT) # Read latest version if no SOURCE_GENERATION supplied
    latest_blob.reload() # Get this objects metadata
    source_generation = int(latest_blob.generation) # Store its generation number as an integer
else:
    source_generation = int(SOURCE_GENERATION) # Else use generation number supplied

source_blob = bucket.blob(SOURCE_OBJECT, generation=source_generation) # Download the file based with the specific generation version value to local storage
source_blob.download_to_filename(str(LOCAL_SOURCE))

# Calculate a unique SHA-256 digital fingerprint for file at the path specified and return it as 64 character hexadecimal string
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_sha256 = sha256_file(LOCAL_SOURCE) # The SHA-256 digital fingerprint generated for the local file will later be saved in our provenance.json file
if EXPECTED_SOURCE_SHA256 is not None:
    assert source_sha256 == EXPECTED_SOURCE_SHA256, (
        "Downloaded file does not match the expected SHA-256."
    )

print("Source SHA-256:", source_sha256)
print("Input:", f"gs://{BUCKET_NAME}/{SOURCE_OBJECT}")
print("Pinned generation:", source_generation)
print("SHA-256:", source_sha256)
print("Downloaded to:", LOCAL_SOURCE)

## 4. Load the processed dataset and do some initial analysis
Load the dataset, display each column and data type and do the following validation checks
* It has the correct number of rows
* There are no missing id's
* There are no duplicate id's
* There is no missing values in the TARGET_COLUMN i.e. delayed_15min column and it only contains values 0 or 1
* The souce_year column contains only years between 2023 and 2025

In [ ]:
df = pd.read_parquet(LOCAL_SOURCE)
print("Dataset shape:", df.shape)
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

# Display first 5 rows
display(df.head())

# Display column names and data types
column_summary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notna().sum().values,
    "Missing Count": df.isna().sum().values
})
display(column_summary)

# Display overall dataset information
df.info()

assert len(df) == 100_000, f"Expected 100,000 rows, got {len(df):,}"
assert df["id"].notna().all(), "Missing flight IDs"
assert df["id"].is_unique, "Flight IDs are not unique"
assert df[TARGET_COLUMN].notna().all(), "Missing target values"
assert set(df[TARGET_COLUMN].unique()).issubset({0, 1}), "Unexpected target classes"
assert df[SOURCE_YEAR_COLUMN].between(2023, 2025).all(), ("Flight year outside expected range")
print("Input validation passed.")

## 5. Completeness

Check for missing values

In [ ]:
# Calculate the number of missing values in each column
missing_count = df.isna().sum()

# Calculate the percentage of missing values
missing_percentage = (df.isna().mean() * 100).round(2)

# Combine the results into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

# Show columns with missing values, largest first
missing_summary = (
    missing_summary[missing_summary["Missing Count"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

display(missing_summary)

## 6. Uniqueness

Check for duplicate rows and duplicate id's



In [ ]:
# Count completely identical rows
exact_duplicates = df.duplicated().sum()

# Count duplicated flight IDs
duplicate_ids = df["id"].duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate flight IDs:", duplicate_ids)

## 7. Validity

Ensure that dataset doesn't contain values that should not be possible


In [ ]:
# Check target values
print("Target values:")
display(df[TARGET_COLUMN].value_counts(dropna=False))

# Check flight years
print("Flight years:")
display(df[SOURCE_YEAR_COLUMN].value_counts(dropna=False).sort_index())

# Check expected flight duration
invalid_duration = (
    df[EXPECTED_DURATION_MIN_COLUMN].notna()
    & (df[EXPECTED_DURATION_MIN_COLUMN] <= 0)
)

print("Non-positive expected durations:", invalid_duration.sum())

## 8. Consistency

Check consistency between related columns

In [ ]:
# Convert flight date into a datetime
flight_dates = pd.to_datetime(df["dof"], errors="coerce")

# Compare year in flight date with source_year
year_mismatch = (
    flight_dates.notna()
    & df[SOURCE_YEAR_COLUMN].notna()
    & (flight_dates.dt.year != df[SOURCE_YEAR_COLUMN])
)

print("Year mismatches:", year_mismatch.sum())
print("Unparseable or missing flight dates:", flight_dates.isna().sum())


# Compare delay_min with delayed_15min
calculated_target = (df["delay_min"] >= 15).astype(int)
mismatches = (calculated_target != df["delayed_15min"]).sum()
mismatch_percentage = (mismatches / len(df));
print("Target mismatches:", mismatches)
print("Target mismatch percentage :", round(mismatch_percentage * 100, 2))

### 8.1. Investigate mismatches

Investigate the mismatches between the delay_min and delayed_15min values


In [ ]:
mismatch_mask = (
    (df["delay_min"] >= 15).astype(int)
    != df["delayed_15min"]
)

mismatch_df = df.loc[
    mismatch_mask,
    [
        "id",
        "delay_min",
        "delayed_15min",
        "expected_duration_min",
        "duration_min"
    ]
].copy()

# Add the target we calculated
mismatch_df["calculated_target"] = (
    mismatch_df["delay_min"] >= 15
).astype(int)

# Calculate the difference between actual and expected duration
mismatch_df["calculated_delay"] = (
    mismatch_df["duration_min"]
    - mismatch_df["expected_duration_min"]
)

display(
    mismatch_df.sort_values("delay_min")
)

### 8.2. Re Assess mismatches using delay_min > 15 rather than >= 15

Based on the mistmatch analysis done above, reassess them using delay_min > 15 rather than >=15

In [ ]:
# Compare delay_min with delayed_15min
calculated_target = (df["delay_min"] > 15).astype(int)
mismatches = (calculated_target != df["delayed_15min"]).sum()
mismatch_percentage = (mismatches / len(df));
print("Target mismatches:", mismatches)
print("Target mismatch percentage :", round(mismatch_percentage * 100, 2))

assert (
    (df["delay_min"] > 15).astype(int)
    == df["delayed_15min"]
).all(), "Target values do not match the verified >15-minute rule"

print("Target validation passed: delay_min > 15")

## 9. Timeliness

Ensure only expected flights from 2023-2025 are included


In [ ]:
print("Earliest flight:", flight_dates.min())
print("Latest flight:", flight_dates.max())

# Number of flights by year
display(
    df.groupby("source_year")
      .size()
      .reset_index(name="Flight Count")
)

## 10. Categorical Feature Analysis

Check how many distinct values the categorical columns contain. This helps us identify high cardinality features (columns with many unique values) which is important when choosing an appropriate encoding strategy.

In [ ]:
# Count unique values in each categorical column
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        df[col].nunique(dropna=True)
        for col in categorical_columns
    ],
    "Missing Percentage": [
        round(df[col].isna().mean() * 100, 2)
        for col in categorical_columns
    ]
})

display(
    categorical_summary.sort_values(
        "Unique Values",
        ascending=False
    )
)

## 11. Descriptive Statistics

Produce the mean, standard deviation, minimum, quartiles, median and maximum for the numerical duration related columns




In [ ]:
numeric_columns = [
    "expected_duration_min",
    "duration_min",
    "delay_min"
]

# Calculate summary statistics
numeric_summary = df[numeric_columns].describe().T

# Add the median explicitly for clarity
numeric_summary["median"] = df[numeric_columns].median()

# Add missing-value counts
numeric_summary["missing_count"] = (
    df[numeric_columns].isna().sum()
)

display(numeric_summary.round(2))

## 12. Calculate Potential Outliers using IQR

IQR measures how spread out the middle 50% of data is.

An IQR outlier is a value outside the range defined by the first and third quartiles, its not automatically an error but usually needs investigation to confirm.

Using Tukeys 1.5 x IQR rule

In [ ]:
numeric_columns = [
    "expected_duration_min",
    "duration_min",
    "delay_min"
]

outlier_results = []

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    lower_outliers = (
        (df[column] < lower_bound)
    )

    upper_outliers = (
        (df[column] > upper_bound)
    )

    outlier_results.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": round(IQR, 2),
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Outlier Count": outliers.sum(),
        "Outlier Percentage": round(outliers.mean() * 100, 2),
        "Lower Outlier Count": lower_outliers.sum(),
        "Lower Outlier Percentage": round(lower_outliers.mean() * 100, 2),
        "Upper Outlier Count": upper_outliers.sum(),
        "Upper Outlier Percentage": round(upper_outliers.mean() * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary)

## 13. Create Histograms

This will help us visually assess if the distributions are skewed and whether extreme values are distorting the plots


In [ ]:
import matplotlib.pyplot as plt

for column in numeric_columns:
    plt.figure(figsize=(9, 4))

    plt.hist(df[column].dropna(), bins=60)

    plt.title(f"Distribution of {column}")
    plt.xlabel("Minutes")
    plt.ylabel("Number of Flights")

    plt.tight_layout()
    plt.show()


# Zoomed-in histogram for flight delays

plt.figure(figsize=(9, 4))

plt.hist(
    df["delay_min"].dropna(),
    bins=90,
    range=(-30, 60),
    edgecolor="black"
)

# Add a line showing the classification threshold
plt.axvline(
    x=15,
    color="red",
    linestyle="--",
    label="15-minute threshold"
)

plt.title("Flight Delay Distribution (Zoomed: -30 to +60 Minutes)")
plt.xlabel("Delay (Minutes)")
plt.ylabel("Number of Flights")

plt.legend()
plt.tight_layout()
plt.show()

## 14. Verify how delay_min is calculated

From earlier anlaysis we suspect that delay_min is being calculated from the expected_duration_min and duration_min


In [ ]:
# Check whether recorded delay_min equals actual minus expected duration
calculated_delay = (
    df["duration_min"] - df["expected_duration_min"]
)

difference = (
    df["delay_min"] - calculated_delay
).abs()

print("Maximum absolute difference:", difference.max())
print(
    "Records matching within 0.01 minutes:",
    (difference <= 0.01).sum()
)

## 15. Temporal Analysis

Investigate whether percentage of flights delayed by more than 15 minutes varies across years and months to help identify potential time-related patterns and to determine if year and month should be considered as candidate predictors.



In [ ]:

# Delay rate by year
yearly_delay = (
    df.groupby("source_year", dropna=False)
      .agg(
          total_flights=("delayed_15min", "size"),
          delayed_flights=("delayed_15min", "sum"),
          delay_rate=("delayed_15min", "mean")
      )
      .reset_index()
      .sort_values("source_year")
)

yearly_delay["delay_rate_pct"] = (
    yearly_delay["delay_rate"] * 100
)

display(
    yearly_delay[
        ["source_year", "total_flights",
         "delayed_flights", "delay_rate_pct"]
    ].round(2)
)


# Create bar chart of yearly delay rates
plt.figure(figsize=(8, 4))

bars = plt.bar(
    yearly_delay["source_year"].astype(str),
    yearly_delay["delay_rate_pct"]
)

plt.title("Percentage of Flights Delayed by Year")
plt.xlabel("Year")
plt.ylabel("Flights Delayed More Than 15 Minutes (%)")

plt.ylim(0, yearly_delay["delay_rate_pct"].max() * 1.2)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{bar.get_height():.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()







In [ ]:
# Calculate monthly delay rates
monthly_delay = (
    df.groupby("source_month", dropna=False)
      .agg(
          total_flights=("delayed_15min", "size"),
          delayed_flights=("delayed_15min", "sum"),
          delay_rate=("delayed_15min", "mean")
      )
      .reset_index()
      .sort_values("source_month")
)

monthly_delay["delay_rate_pct"] = (
    monthly_delay["delay_rate"] * 100
)

display(
    monthly_delay[
        ["source_month", "total_flights",
         "delayed_flights", "delay_rate_pct"]
    ].round(2)
)

# Plot the monthly delay rates
plt.figure(figsize=(13, 5))

plt.plot(
    monthly_delay["source_month"].astype(str),
    monthly_delay["delay_rate_pct"],
    marker="o",
    markersize=4
)

plt.title("Monthly Flight Delay Rates (2023–2025)")
plt.xlabel("Year and Month")
plt.ylabel("Flights Delayed More Than 15 Minutes (%)")

plt.xticks(rotation=90)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# Compare the same calendar months across years
temporal_df = df[
    ["source_month", "source_year", "delayed_15min"]
].copy()

temporal_df["calendar_month"] = (
    temporal_df["source_month"]
    .astype(str)
    .str[-2:]
    .astype(int)
)

month_year_delay = (
    temporal_df.groupby(
        ["source_year", "calendar_month"]
    )["delayed_15min"]
    .mean()
    .mul(100)
    .unstack("source_year")
    .sort_index()
)

display(month_year_delay.round(2))

plt.figure(figsize=(10, 5))

for year in month_year_delay.columns:
    plt.plot(
        month_year_delay.index,
        month_year_delay[year],
        marker="o",
        label=str(year)
    )

plt.title("Flight Delay Rates by Calendar Month and Year")
plt.xlabel("Calendar Month")
plt.ylabel("Flights Delayed More Than 15 Minutes (%)")
plt.xticks(range(1, 13))
plt.legend(title="Year")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Identify months with the highest delay rates and display their sample sizes
display(
    monthly_delay[
        [
            "source_month",
            "total_flights",
            "delayed_flights",
            "delay_rate_pct"
        ]
    ]
    .sort_values("delay_rate_pct", ascending=False)
    .head(10)
    .round(2)
)

## 16. Flight Delay Analysis by Depature Airport

Analyse whether the percentage of delayed flights differs between depature airports, comparing the 10 most frequently represented departure airports flight counts, delayed flight counts and delay rates.

In [ ]:

# Calculate delay rates for the 10 busiest departure airports in the sample.
top_10_airports = df["adep"].value_counts().head(10).index

airport_analysis = (
    df[df["adep"].isin(top_10_airports)]
    .groupby("adep")
    .agg(
        total_flights=("id", "count"),
        delayed_flights=("delayed_15min", "sum")
    )
    .reset_index()
)

airport_analysis["delay_rate_pct"] = (
    airport_analysis["delayed_flights"]
    / airport_analysis["total_flights"] * 100
).round(2)

airport_analysis = airport_analysis.sort_values(
    "total_flights", ascending=False
)

display(airport_analysis)


In [ ]:

# Compare airport delay rates visually
import matplotlib.pyplot as plt

overall_delay_rate = df["delayed_15min"].mean() * 100

plot_data = airport_analysis.sort_values(
    "delay_rate_pct", ascending=False
)

plt.figure(figsize=(10, 5))

bars = plt.bar(
    plot_data["adep"],
    plot_data["delay_rate_pct"]
)

plt.axhline(
    overall_delay_rate,
    color="red",
    linestyle="--",
    label=f"Overall sample average: {overall_delay_rate:.2f}%"
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.15,
        f"{height:.2f}%",
        ha="center",
        fontsize=9
    )

plt.title("Flight Delay Rates – 10 Most Frequent Departure Airports")
plt.xlabel("Departure Airport (ICAO Code)")
plt.ylabel("Flights Delayed More Than 15 Minutes (%)")
plt.legend()
plt.ylim(0, max(plot_data["delay_rate_pct"].max(), overall_delay_rate) * 1.2)
plt.tight_layout()
plt.show()


In [ ]:

# Compare airports with at least 100 flights.

all_airports = (
    df.groupby("adep")
    .agg(
        total_flights=("id", "count"),
        delayed_flights=("delayed_15min", "sum")
    )
    .reset_index()
)

all_airports["delay_rate_pct"] = (
    all_airports["delayed_flights"]
    / all_airports["total_flights"] * 100
).round(2)

minimum_flights = 100

eligible_airports = all_airports[
    all_airports["total_flights"] >= minimum_flights
]

print("Total departure airports:", len(all_airports))
print("Airports with at least 100 flights:", len(eligible_airports))

print("\n10 highest delay rates:")
display(
    eligible_airports.nlargest(10, "delay_rate_pct")
)

print("\n10 lowest delay rates:")
display(
    eligible_airports.nsmallest(10, "delay_rate_pct")
)


In [ ]:

# Compare Zurich Airport delay rates across years

zurich_by_year = (
    df[df["adep"] == "LSZH"]
    .groupby("source_year")
    .agg(
        total_flights=("id", "count"),
        delayed_flights=("delayed_15min", "sum")
    )
    .reset_index()
)

zurich_by_year["delay_rate_pct"] = (
    zurich_by_year["delayed_flights"]
    / zurich_by_year["total_flights"] * 100
).round(2)

display(zurich_by_year)


## 17. Aircraft Charactersitics and Flight Delay Rates

Investigate whether different aircraft characteristics are associated with the likelihood of flight being delayed

In [ ]:

# Delay rates by aircraft class

aircraft_class_df = df.copy()

# Keep missing values as a separate category for EDA
aircraft_class_df["aircraft_class_group"] = (
    aircraft_class_df["icao_aircraft_class"]
    .fillna("Missing")
)

aircraft_class_analysis = (
    aircraft_class_df
    .groupby("aircraft_class_group")
    .agg(
        total_flights=("delayed_15min", "size"),
        delayed_flights=("delayed_15min", "sum"),
        delay_rate=("delayed_15min", "mean")
    )
    .reset_index()
)

aircraft_class_analysis["delay_rate_pct"] = (
    aircraft_class_analysis["delay_rate"] * 100
).round(2)

aircraft_class_analysis = aircraft_class_analysis.sort_values(
    "total_flights", ascending=False
)

display(
    aircraft_class_analysis[
        [
            "aircraft_class_group",
            "total_flights",
            "delayed_flights",
            "delay_rate_pct"
        ]
    ]
)


In [ ]:

# Plot aircraft class delay rates

minimum_flights = 100
overall_delay_rate = df["delayed_15min"].mean() * 100

class_plot_data = (
    aircraft_class_analysis[
        aircraft_class_analysis["total_flights"] >= minimum_flights
    ]
    .sort_values("delay_rate_pct", ascending=False)
)

plt.figure(figsize=(11, 6))

bars = plt.barh(
    class_plot_data["aircraft_class_group"],
    class_plot_data["delay_rate_pct"]
)

plt.axvline(
    overall_delay_rate,
    color="red",
    linestyle="--",
    label=f"Overall sample average: {overall_delay_rate:.2f}%"
)

plt.gca().invert_yaxis()

plt.title("Flight Delay Rates by Aircraft Class")
plt.xlabel("Flights Delayed More Than 15 Minutes (%)")
plt.ylabel("Aircraft Class")

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Compare known and missing aircraft classes

missing_class_analysis = (
    df.assign(
        class_status=df["icao_aircraft_class"]
        .isna()
        .map({
            True: "Missing aircraft class",
            False: "Known aircraft class"
        })
    )
    .groupby("class_status")
    .agg(
        total_flights=("delayed_15min", "size"),
        delayed_flights=("delayed_15min", "sum"),
        delay_rate=("delayed_15min", "mean")
    )
    .reset_index()
)

missing_class_analysis["delay_rate_pct"] = (
    missing_class_analysis["delay_rate"] * 100
).round(2)

display(
    missing_class_analysis[
        [
            "class_status",
            "total_flights",
            "delayed_flights",
            "delay_rate_pct"
        ]
    ]
)


In [ ]:

# Analyse the 10 most frequent aircraft typecodes

top_10_typecodes = (
    df["typecode"]
    .value_counts()
    .head(10)
    .index
)

typecode_analysis = (
    df[df["typecode"].isin(top_10_typecodes)]
    .groupby("typecode")
    .agg(
        total_flights=("delayed_15min", "size"),
        delayed_flights=("delayed_15min", "sum"),
        delay_rate=("delayed_15min", "mean")
    )
    .reset_index()
)

typecode_analysis["delay_rate_pct"] = (
    typecode_analysis["delay_rate"] * 100
).round(2)

typecode_analysis = typecode_analysis.sort_values(
    "total_flights", ascending=False
)

display(
    typecode_analysis[
        [
            "typecode",
            "total_flights",
            "delayed_flights",
            "delay_rate_pct"
        ]
    ]
)


In [ ]:

# Plot the 10 most frequent aircraft types

typecode_plot_data = typecode_analysis.sort_values(
    "delay_rate_pct", ascending=False
)

plt.figure(figsize=(10, 5))

bars = plt.bar(
    typecode_plot_data["typecode"],
    typecode_plot_data["delay_rate_pct"]
)

plt.axhline(
    overall_delay_rate,
    color="red",
    linestyle="--",
    label=f"Overall sample average: {overall_delay_rate:.2f}%"
)

plt.title("Flight Delay Rates – 10 Most Frequent Aircraft Types")
plt.xlabel("Aircraft Typecode")
plt.ylabel("Flights Delayed More Than 15 Minutes (%)")
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

# Compare known and missing aircraft typecodes

typecode_missing_analysis = (
    df.assign(
        typecode_status=df["typecode"]
        .isna()
        .map({
            True: "Missing typecode",
            False: "Known typecode"
        })
    )
    .groupby("typecode_status")
    .agg(
        total_flights=("delayed_15min", "size"),
        delayed_flights=("delayed_15min", "sum"),
        delay_rate=("delayed_15min", "mean")
    )
    .reset_index()
)

typecode_missing_analysis["delay_rate_pct"] = (
    typecode_missing_analysis["delay_rate"] * 100
).round(2)

display(
    typecode_missing_analysis[
        [
            "typecode_status",
            "total_flights",
            "delayed_flights",
            "delay_rate_pct"
        ]
    ]
)


In [ ]:

# Check whether aircraft class and typecode
# are missing on the same flight records.

missing_overlap = pd.crosstab(
    df["icao_aircraft_class"].isna(),
    df["typecode"].isna(),
    rownames=["Aircraft Class Missing"],
    colnames=["Typecode Missing"],
    margins=True
)

display(missing_overlap)

# Calculate number of records missing both fields
both_missing = (
    df["icao_aircraft_class"].isna()
    & df["typecode"].isna()
).sum()

print("Records missing both fields:", both_missing)
print(
    f"Percentage of sample: {both_missing / len(df) * 100:.2f}%"
)
